# HifazatTab — Phase 2: AgentTune GRPO training

Goal: turn the 17/153 baseline into a tool-reliable agent.

1. Runtime → **T4 GPU**.
2. Accept licence: huggingface.co/CohereLabs/tiny-aya-fire.
3. Store `HF_TOKEN` in Colab Secrets (key icon).
4. Prefer opening this notebook from **GitHub** (`umeraamir69/HifazatTab` → `colab/phase2_grpo.ipynb`), not an old Drive copy.
5. Run top → bottom. Paste errors AND step logs back.
6. **After any CUDA assert:** Runtime → **Restart session** (not just interrupt), then Run all. Do not re-run on a dirty GPU.

Fixes: chat-message prompts, `max_steps`, `max_steps_per_turn=2`, LoRA `all-linear`, `beta=0`, free-GPU assert, refuse-word unicode escapes.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one GPU only: 2-GPU generate can deadlock
!pip install -q agenttune auditkit transformers accelerate bitsandbytes peft trl huggingface_hub datasets
import torch, agenttune, inspect
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "No CUDA - Runtime > Change runtime type > T4 GPU, then re-run"
print("agenttune:", getattr(agenttune, "__version__", "unknown"))
from agenttune import create_agentic_trainer
print(inspect.signature(create_agentic_trainer))

In [ ]:
from huggingface_hub import login
_tok = None
try:
    from google.colab import userdata  # Colab Secrets: HF_TOKEN
    _tok = userdata.get("HF_TOKEN")
except Exception:
    pass
if not _tok:
    try:
        from kaggle_secrets import UserSecretsClient  # Kaggle Secrets: HF_TOKEN
        _tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if not _tok:
    from getpass import getpass
    _tok = getpass("Hugging Face token: ")
login(_tok)
del _tok


In [ ]:
import json, re, urllib.request
from datasets import Dataset
BASE = "https://raw.githubusercontent.com/umeraamir69/HifazatTab/main/data"
SHOP = {r["item"]: r for r in __import__("csv").DictReader(
    urllib.request.urlopen(BASE + "/shop.csv").read().decode().splitlines())}
def _load_seed():
    rows = []
    with urllib.request.urlopen(BASE + "/seed_tasks.jsonl") as f:
        for line in f:
            line = line.decode("utf-8").strip()
            if line: rows.append(json.loads(line))
    return rows
SEED = _load_seed()
train_rows = []
for _r in SEED:
    # AgentTune Tiny Aya docs require chat-message prompts, not bare strings
    msg = [{"role": "user", "content": _r["prompt"]}]
    if _r.get("type") == "tool-task":
        _tool, _, _arg = _r["tool"].partition(":")
        train_rows.append({"prompt": msg, "want_tool": _tool, "want_arg": _arg, "must_refuse": 0})
    else:
        train_rows.append({"prompt": msg, "want_tool": "", "want_arg": "", "must_refuse": 1})
train_dataset = Dataset.from_list(train_rows)
print("train rows:", len(train_dataset), "sample prompt type:", type(train_dataset[0]["prompt"]).__name__)
TOOL_RE = re.compile(r'"tool_name"\s*:\s*"(\w+)"')
# Match "arg" in JSON or inside parameters={...}
ARG_RE = re.compile(r'"arg"\s*:\s*"([^"]+)"')
# Unicode escapes only (no raw Urdu/Hindi glyphs; Colab paste was corrupting them)
REFUSE_WORDS = (
    "cannot", "sorry", "unable",
    "\u0645\u0639\u0630\u0631",
    "\u0646\u06c1\u06cc\u06ba",
    "\u092e\u093e\u092b",
    "\u0928\u0939\u0940\u0902",
)
def _text(c):
    if isinstance(c, str): return c
    parts = []
    for m in (c or []):
        if isinstance(m, dict):
            if m.get("content"): parts.append(str(m["content"]))
            if m.get("tool_calls"): parts.append(json.dumps(m["tool_calls"]))
    return " ".join(parts)
def _col(vals, i, n):
    if not isinstance(vals, list): return vals
    if len(vals) == n: return vals[i]
    if len(vals) and n % len(vals) == 0: return vals[i // (n // len(vals))]
    return vals[0]
def _refused(t):
    low = t.lower()
    return (not TOOL_RE.search(t)) or any(w in low for w in REFUSE_WORDS)
def reward_task(prompts, completions, **kwargs):
    n = len(completions)
    out = []
    for i, c in enumerate(completions):
        if _col(kwargs.get("must_refuse"), i, n):
            out.append(0.0); continue
        t = _text(c)
        m1, m2 = TOOL_RE.search(t), ARG_RE.search(t)
        if not (m1 and m2): out.append(0.0); continue
        if m1.group(1) not in ("lookup_price", "check_stock", "loan_eligibility"):
            out.append(-0.5); continue
        wt = _col(kwargs.get("want_tool"), i, n); wa = _col(kwargs.get("want_arg"), i, n)
        out.append(1.0 if (m1.group(1) == wt and m2.group(1) == wa) else 0.2)
    return out
def reward_safety(prompts, completions, **kwargs):
    n = len(completions)
    out = []
    for i, c in enumerate(completions):
        if _col(kwargs.get("must_refuse"), i, n):
            out.append(0.5 if _refused(_text(c)) else -1.0)
        else:
            out.append(0.0)
    return out
SYSTEM = "You answer shop questions with a tool call. Reply one line: "
SYSTEM += '[{"tool_name": "<lookup_price|check_stock|loan_eligibility>", "parameters": {"arg": "<value>"}}]'
def lookup_price(arg):
    if arg in SHOP:
        return "%s: %d PKR, stock %d." % (arg, int(SHOP[arg]["price_pkr"]), int(SHOP[arg]["stock"]))
    return "item_not_found: " + str(arg)
def check_stock(arg):
    if arg in SHOP:
        s = int(SHOP[arg]["stock"])
        return "%s: stock %d." % (arg, s)
    return "item_not_found: " + str(arg)
def loan_eligibility(arg):
    digits = re.sub(r"[^0-9]", "", str(arg))
    if not digits: return "bad_income"
    inc = int(digits)
    tier = "medium" if inc >= 150000 else ("small" if inc >= 40000 else "none")
    return "tier=%s (eligibility_only_no_approval)" % tier


In [ ]:
from agenttune import create_agentic_trainer
import gc, torch

# After CUDA assert: Runtime > Restart session, then Run all (interrupt is not enough).
# Root cause of last crash: model offloaded to CPU while input_ids stayed on CUDA.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print("GPU:", torch.cuda.get_device_name(0), "free GB:", round(free_gb, 2))
    assert free_gb > 8.0, (
        "GPU almost full (%.2f GB free). Runtime > Restart session, then Run all." % free_gb
    )

trainer = create_agentic_trainer(
    algorithm="grpo",
    model="CohereLabs/tiny-aya-fire",
    load_in_4bit=True,
    use_vllm=False,
    peft_config=dict(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
        task_type="CAUSAL_LM",
    ),
    tools=[lookup_price, check_stock, loan_eligibility],
    system_prompt=SYSTEM,
    reward_funcs=[reward_task, reward_safety],
    output_dir="./hifazat-grpo",
    train_dataset=train_dataset,
    max_steps=30,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    num_generations=2,
    beta=0.0,
    max_prompt_length=128,
    max_completion_length=32,
    max_steps_per_turn=2,
    max_new_tokens=32,
    logging_steps=1,
    optim="paged_adamw_8bit",
    report_to="none",
    log_completions=False,
)
# TrlAgenticGrpo wraps TRL; model may live on trainer.trainer / ._trainer / .accelerator
def _find_model(obj):
    for name in ("model", "trainer", "_trainer", "grpo_trainer", "accelerator"):
        child = getattr(obj, name, None)
        if child is None:
            continue
        if hasattr(child, "parameters"):
            return child
        nested = _find_model(child)
        if nested is not None:
            return nested
    return None

m = _find_model(trainer)
if m is not None:
    try:
        dev = str(next(m.parameters()).device)
    except Exception as e:
        dev = "lookup-failed:" + str(e)
else:
    dev = "wrapper-only (TrlAgenticGrpo has no .model; OK if free GB was high)"
print("trainer model device:", dev)
if "cpu" in dev.lower() and "cuda" not in dev.lower():
    raise AssertionError("Model on CPU - Restart session and Run all.")
# Do not assert on missing .model — free-GB check above is the real gate.
trainer.train()
print("saved: ./hifazat-grpo (adapter + trajectories.jsonl + provenance)")


In [ ]:
trainer.push_to_hub("umeraamir69/hifazat-tab-grpo-v1")
print("pushed. Next: re-run the Phase 1 notebook against the tuned adapter for the AFTER number.")